# Part 2 — Visualising Distributions
### Data Mondays, Week 4 — Amani Insurance claims case study

**Deliverable:** visualise distributions using **Matplotlib** and **Seaborn**.

Every chart below is displayed inline with `plt.show()` *and* saved as a PNG into
a `plots/` folder, in case you want them for slides afterwards.

We reuse the same light cleaning from Part 1 — full cleanup is Part 3's job.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

FILE_PATH = "insurance_claims_messy.csv"
PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")  # a clean, readable default look for every Seaborn chart


def save_and_show(fig, name):
    """Save a Matplotlib figure to plots/ and display it inline."""
    path = os.path.join(PLOTS_DIR, name)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    print(f"saved -> {path}")
    plt.show()
    plt.close(fig)

In [ ]:
# Same light cleaning as Part 1: make claim_amount_kes numeric, and
# canonicalise claim_type/status just enough for grouping and colouring.
df = pd.read_csv(FILE_PATH)
df["claim_amount_kes"] = (
    df["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce")
df["claim_type_clean"] = (
    df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
)
df["status_clean"] = df["status"].astype(str).str.strip().str.lower()

# usable: only claims with a valid, positive amount -- charts about SIZE
# of claims need a real number to plot.
usable = df[df["claim_amount_kes"] > 0].copy()

## Section 1 — A basic Matplotlib histogram

A histogram groups values into "bins" (ranges) and counts how many claims fall
into each one — the classic way to see a distribution's overall shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(usable["claim_amount_kes"], bins=40, color="#3b6fa0", edgecolor="white")
ax.set_title("Distribution of claim amounts (KES)")
ax.set_xlabel("Claim amount (KES)")
ax.set_ylabel("Number of claims")
save_and_show(fig, "01_matplotlib_histogram.png")

**Reading this chart:** the bars pile up hard on the left with a long, thin tail
stretching to the right. That's the right-skew we calculated numerically in
Part 1 (mean > median), now visible directly.

## Section 2 — The same distribution, log-scaled x-axis

When a few extreme values are much larger than the rest, a normal ("linear")
axis squashes most of the data into one tall bar near zero. A **log scale**
spaces out small and large values more evenly, revealing detail that the linear
version hides.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(usable["claim_amount_kes"], bins=40, color="#3b6fa0", edgecolor="white")
ax.set_xscale("log")   # the only change from Section 1 -- log-scale the x-axis
ax.set_title("Distribution of claim amounts (log scale)")
ax.set_xlabel("Claim amount (KES, log scale)")
ax.set_ylabel("Number of claims")
save_and_show(fig, "02_matplotlib_histogram_logscale.png")

**Reading this chart:** on a log scale, both the small, frequent health-outpatient
claims *and* the huge, rare property-fire claims become visible in the same
picture. Right-skewed financial data (claim sizes, incomes, house prices) is
almost always easier to read on a log axis.

## Section 3 — Seaborn: distributions split by claim type

Seaborn is built on top of Matplotlib, but handles a lot of the fiddly work
(colours, legends, grouping) automatically. Here we overlay one distribution
curve per claim type, on log-scaled axes, in a handful of lines.

In [ ]:
fig = plt.figure(figsize=(9, 5))
ax = sns.histplot(
    data=usable, x="claim_amount_kes", hue="claim_type_clean",  # hue = one colour per category
    log_scale=True, element="step", common_norm=False,
)
ax.set_title("Claim amount distribution by claim type (log scale)")
save_and_show(ax.figure, "03_seaborn_histplot_by_type.png")

**Reading this chart:** compare the *width* of each colour's curve. A narrow
curve means that claim type's amounts are fairly predictable; a wide curve means
they vary a lot. This lines up with the `std` column from Part 1's groupby
table — wider curve here should mean larger std there.

## Section 4 — Boxplots: built specifically to show outliers

A boxplot draws a box from Q1 to Q3 (the middle 50% of the data), a line at the
median, "whiskers" extending to the typical range, and individual dots for any
value that falls outside that range — i.e. a statistical outlier.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Order claim types by median (highest first), purely so the chart reads
# left-to-right from biggest to smallest typical claim.
order = (
    usable.groupby("claim_type_clean")["claim_amount_kes"].median()
    .sort_values(ascending=False).index
)
sns.boxplot(data=usable, x="claim_type_clean", y="claim_amount_kes", order=order, ax=ax)
ax.set_yscale("log")
ax.set_title("Claim amount by type -- boxplot (log scale)")
ax.set_xlabel("Claim type")
ax.set_ylabel("Claim amount (KES, log scale)")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")  # angle the labels so they don't overlap
save_and_show(fig, "04_seaborn_boxplot_outliers.png")

**Reading this chart:** every dot above the top whisker is a statistical outlier
*for that claim type specifically* — not for the dataset as a whole. A KES
40,000 outpatient claim and a KES 40,000 motor claim mean very different things,
which is exactly why we check outliers per group rather than across everything
at once. This picture is the visual version of the IQR method we'll formalise
with code in Part 3.

## Section 5 — Countplots: distributions of categorical columns

Histograms and boxplots are for *numbers*. A **countplot** answers a different
kind of question for *categories*: "how many of each do we have?" — here, claims
by region and by status.

In [ ]:
# Canonicalise region the same lightweight way, just for this chart.
df["region_clean"] = df["region"].astype(str).str.strip().str.title()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=df, x="region_clean", ax=axes[0],
              order=df["region_clean"].value_counts().index)
axes[0].set_title("Claims by region")
axes[0].set_xlabel("Region")
plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")

sns.countplot(data=df, x="status_clean", ax=axes[1],
              order=df["status_clean"].value_counts().index)
axes[1].set_title("Claims by status")
axes[1].set_xlabel("Status")
plt.setp(axes[1].get_xticklabels(), rotation=20, ha="right")

fig.tight_layout()
save_and_show(fig, "05_seaborn_countplots.png")

**Reading this chart:** notice `status_clean` was lower-cased before plotting —
without that step, `"approved"` / `"Approved"` / `"APPROVED"` would each get
their own bar instead of being combined into one. That's exactly the kind of
categorical messiness Part 3 fixes properly, before any real reporting happens.

**Up next:** Part 3 formalises what we've only *seen* so far — missing values
and outliers — into a reusable cleaning function.